# imbalanced classification (SMOTE)

use imbalanced-learn. construct a 95/5 imbalance from breast cancer.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

rng = np.random.RandomState(0)
data = load_breast_cancer()
X, y = data.data, data.target
# downsample positive class to make imbalance
pos = np.where(y == 1)[0]
neg = np.where(y == 0)[0]
keep = rng.choice(pos, size=20, replace=False)
idx = np.concatenate([neg, keep])
X, y = X[idx], y[idx]
print('class counts:', np.bincount(y))

## baseline (no resampling)

In [2]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
lr = LogisticRegression(solver='lbfgs', max_iter=2000).fit(Xtr, ytr)
print(classification_report(yte, lr.predict(Xte)))

## with SMOTE on training set only

In [3]:
sm = SMOTE(random_state=0)
Xs, ys = sm.fit_resample(Xtr, ytr)
print('after smote:', np.bincount(ys))
lr2 = LogisticRegression(solver='lbfgs', max_iter=2000).fit(Xs, ys)
print(classification_report(yte, lr2.predict(Xte)))

## comparison: class_weight='balanced'

In [4]:
lr3 = LogisticRegression(solver='lbfgs', max_iter=2000, class_weight='balanced').fit(Xtr, ytr)
print(classification_report(yte, lr3.predict(Xte)))

## with imblearn Pipeline (CV-safe)

In [5]:
from imblearn.pipeline import Pipeline as ImbPipe
from sklearn.model_selection import cross_val_score
p = ImbPipe([('sm', SMOTE(random_state=0)),
             ('lr', LogisticRegression(solver='lbfgs', max_iter=2000))])
cross_val_score(p, X, y, cv=5).mean()

## ADASYN and BorderlineSMOTE

In [6]:
from imblearn.over_sampling import ADASYN, BorderlineSMOTE
for sampler in [SMOTE(random_state=0), ADASYN(random_state=0), BorderlineSMOTE(random_state=0)]:
    p = ImbPipe([('s', sampler), ('lr', LogisticRegression(solver='lbfgs', max_iter=2000))])
    s = cross_val_score(p, X, y, cv=5).mean()
    print(f'{type(sampler).__name__:18s}  {s:.4f}')